In [ ]:
from pathlib import Path
import numpy as np
from sunpy.net import Fido, attrs as a
from concurrent.futures import ThreadPoolExecutor
import logging
import xarray as xr
import pandas as pd
import time

class SXRDownloader:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    def __init__(self, save_dir: str = '/downloads/goes_data', concat_dir: str = '/downloads/goes_combined'):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
        self.concat_dir = Path(concat_dir)
        self.concat_dir.mkdir(exist_ok=True)
        self.used_g13_files = []
        self.used_g14_files = []
        self.used_g15_files = []
        self.used_g16_files = []
        self.used_g17_files = []
        self.used_g18_files = []

    def download_and_save_goes_data(self, start='2023-07-01', end='2023-08-15', max_workers=4):
        """
        Download GOES X-ray data at 1-minute cadence for a specified time range, including all available satellites.

        Parameters:
        - start (str): Start date for the query (e.g., '2023-07-01')
        - end (str): End date for the query (e.g., '2023-08-15')
        - max_workers (int): Number of parallel download threads
        """
        logging.info(f"Searching GOES X-ray data from {start} to {end} for all satellites at 1-minute cadence...")

        # Query for all GOES satellites with 1-minute averaged XRS data
        goes_query = Fido.search(
            a.Time(start, end),
            a.Instrument('XRS'),
            a.Resolution('avg1m')  # 1-minute averaged data
        )

        logging.info(f"Found {len(goes_query)} GOES files.")

        # Skip if no files found
        if len(goes_query) == 0:
            logging.warning("No files found for the specified query.")
            return []

        # Define download function for a single file
        def download_file(file_entry, path_template):
            try:
                fido_result = Fido.fetch(file_entry, path=str(path_template / "{file}"))
                return fido_result
            except Exception as e:
                logging.error(f"Failed to download {file_entry['file']}: {e}")
                return []

        # Use ThreadPoolExecutor for parallel downloads
        logging.info(f"Downloading {len(goes_query[0])} files with {max_workers} workers...")
        downloaded_files = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Create download tasks for each file
            futures = [
                executor.submit(download_file, row, self.save_dir)
                for row in goes_query[0]
            ]
            # Collect results
            for future in futures:
                result = future.result()
                downloaded_files.extend(result)

        logging.info(f"Saved {len(downloaded_files)} files to {self.save_dir}")
        return downloaded_files

In [ ]:
sxr = SXRDownloader('/mnt/data/Checking_GOES','/mnt/data/Checking_GOES/Combined')

In [ ]:
sxr.download_and_save_goes_data('2012-01-01','2017-02-28',max_workers=10)

In [ ]:
import tqdm
import argparse
import logging
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

class SXRDataProcessor:
    """Class to process GOES X-ray data, including downloading, combining, and interpolating data from multiple satellites.
    This class handles the downloading of GOES data, combining data from different satellites, and applying interpolation
    in log space to the X-ray flux data.
    It also tracks which files were used in the processing.
    Parameters
    ----------
    save_dir : str
        Directory where downloaded GOES data will be saved.
    concat_dir : str
        Directory where combined GOES data will be saved.
    """

    def __init__(self, data_dir: str = '/mnt/data/AUGUST/GOES-timespan', output_dir: str = '/mnt/data/AUGUST/combined'):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.used_g13_files = []
        self.used_g14_files = []
        self.used_g15_files = []
        self.used_g16_files = []
        self.used_g17_files = []
        self.used_g18_files = []

    def combine_goes_data(self, columns_to_interp=["xrsb_flux", "xrsa_flux"]):
        """
        Combine GOES-16 and GOES-18 files and track source files used.
        Parameters
        """
        print("🔍 Scanning for GOES data files...")
        
        g13_files = sorted(self.data_dir.glob("*g13*.nc"))
        g14_files = sorted(self.data_dir.glob("*g14*.nc"))
        g15_files = sorted(self.data_dir.glob("*g15*.nc"))
        g16_files = sorted(self.data_dir.glob("*g16*.nc"))
        g17_files = sorted(self.data_dir.glob("*g17*.nc"))
        g18_files = sorted(self.data_dir.glob("*g18*.nc"))
        
        total_files = len(g13_files) + len(g14_files) + len(g15_files) + len(g16_files) + len(g17_files) + len(g18_files)
        logging.info(
            f"Found {len(g13_files)} GOES-13 files, {len(g14_files)} GOES-14 files, {len(g15_files)} GOES-15 files, {len(g16_files)} GOES-16 files, {len(g17_files)} GOES-17 files, and {len(g18_files)} GOES-18 files.")
        print(f"📊 Total files found: {total_files}")
        
        if total_files == 0:
            print("⚠️  No GOES data files found in the specified directory.")
            return

        def process_files(files, satellite_name, output_file, used_file_list):
            datasets = []
            combined_meta = {}
            successful_files = 0
            failed_files = 0

            print(f"🛰️  Processing {satellite_name} ({len(files)} files)...")
            
            # Progress bar for file loading
            with tqdm(files, desc=f"Loading {satellite_name}", unit="file", 
                     bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:
                for file_path in pbar:
                    pbar.set_postfix_str(f"Loading {file_path.name}")
                    try:
                        ds = xr.open_dataset(str(file_path))
                        datasets.append(ds)
                        used_file_list.append(file_path)  # Track file used
                        successful_files += 1
                        logging.info(f"Loaded {file_path.name}")
                    except Exception as e:
                        failed_files += 1
                        logging.error(f"Could not load {file_path.name}: {e}")
                        continue
                    finally:
                        if 'ds' in locals():
                            ds.close()

            if not datasets:
                print(f"❌ No valid datasets for {satellite_name}")
                logging.warning(f"No valid datasets for {satellite_name}")
                return

            print(f"📊 Processing {len(datasets)} datasets for {satellite_name}...")
            
            try:
                print(f"🔗 Concatenating datasets...")
                combined_ds = xr.concat(datasets, dim='time').sortby('time')
                
                # Scaling factors for GOES-13, GOES-14, and GOES-15
                if satellite_name in ['GOES-13', 'GOES-14', 'GOES-15']:
                    print(f"⚖️  Applying scaling factors for {satellite_name}...")
                    combined_ds['xrsa_flux'] = combined_ds['xrsa_flux'] / .85
                    combined_ds['xrsb_flux'] = combined_ds['xrsb_flux'] / .7
                
                print(f"🔄 Converting to DataFrame...")
                df = combined_ds.to_dataframe().reset_index()
                
                if 'quad_diode' in df.columns:
                    print(f"🔍 Filtering quad diode data...")
                    df = df[df['quad_diode'] == 0]  # Filter out quad diode data
                
                df['time'] = pd.to_datetime(df['time'])
                df.set_index('time', inplace=True)
                
                print(f"📈 Applying log interpolation...")
                df_log = np.log10(df[columns_to_interp].replace(0, np.nan))

                # Step 3: Interpolate in log space
                df_log_interp = df_log.interpolate(method="time", limit_direction="both")

                # Step 4: Back-transform to linear space
                df[columns_to_interp] = 10 ** df_log_interp

                # Add min and max dates to filename
                min_date = df.index.min().strftime('%Y%m%d')
                max_date = df.index.max().strftime('%Y%m%d')
                filename = f"{str(output_file)}_{min_date}_{max_date}.csv"
                
                print(f"💾 Saving to {filename}...")
                df.to_csv(filename, index=True)

                print(f"✅ Successfully processed {satellite_name}: {successful_files} files loaded, {failed_files} failed")
                logging.info(f"Saved combined file: {output_file}")
                
            except Exception as e:
                print(f"❌ Failed to process {satellite_name}: {e}")
                logging.error(f"Failed to write {output_file}: {e}")
            finally:
                for ds in datasets:
                    ds.close()

        # Create list of satellites to process
        satellites_to_process = []
        if len(g13_files) != 0:
            satellites_to_process.append((g13_files, "GOES-13", self.output_dir / "combined_g13_avg1m", self.used_g13_files))
        if len(g14_files) != 0:
            satellites_to_process.append((g14_files, "GOES-14", self.output_dir / "combined_g14_avg1m", self.used_g14_files))
        if len(g15_files) != 0:
            satellites_to_process.append((g15_files, "GOES-15", self.output_dir / "combined_g15_avg1m", self.used_g15_files))
        if len(g16_files) != 0:
            satellites_to_process.append((g16_files, "GOES-16", self.output_dir / "combined_g16_avg1m", self.used_g16_files))
        if len(g17_files) != 0:
            satellites_to_process.append((g17_files, "GOES-17", self.output_dir / "combined_g17_avg1m", self.used_g17_files))
        if len(g18_files) != 0:
            satellites_to_process.append((g18_files, "GOES-18", self.output_dir / "combined_g18_avg1m", self.used_g18_files))

        print(f"\n🚀 Starting processing of {len(satellites_to_process)} satellites...")
        
        # Process each satellite with overall progress tracking
        successful_satellites = 0
        failed_satellites = 0
        
        for i, (files, satellite_name, output_file, used_file_list) in enumerate(satellites_to_process, 1):
            print(f"\n{'='*60}")
            print(f"📡 Processing satellite {i}/{len(satellites_to_process)}: {satellite_name}")
            print(f"{'='*60}")
            
            try:
                process_files(files, satellite_name, output_file, used_file_list)
                successful_satellites += 1
            except Exception as e:
                print(f"❌ Failed to process {satellite_name}: {e}")
                failed_satellites += 1
                logging.error(f"Failed to process {satellite_name}: {e}")
        
        # Print final summary
        print(f"\n{'='*60}")
        print(f"📊 PROCESSING COMPLETE")
        print(f"{'='*60}")
        print(f"✅ Successfully processed: {successful_satellites} satellites")
        print(f"❌ Failed: {failed_satellites} satellites")
        print(f"📁 Total files processed: {total_files}")
        print(f"📂 Output directory: {self.output_dir}")
        
        # Print file usage statistics
        total_used_files = (len(self.used_g13_files) + len(self.used_g14_files) + 
                           len(self.used_g15_files) + len(self.used_g16_files) + 
                           len(self.used_g17_files) + len(self.used_g18_files))
        print(f"📋 Files used in processing: {total_used_files}")
        
        if successful_satellites > 0:
            print(f"\n🎉 SXR data processing completed successfully!")
        else:
            print(f"\n⚠️  No satellites were processed successfully.")

In [ ]:
dp = SXRDataProcessor('/mnt/data/Checking_GOES','/mnt/data/Checking_GOES/Combined/')

In [ ]:
dp.combine_goes_data()

In [ ]:
import pandas as pd
dat18 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g18_avg1m_20220902_20250930.csv')
dat17 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g17_avg1m_20180601_20240811.csv')
dat16 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g16_avg1m_20170207_20250406.csv')
dat15 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g15_avg1m_20120101_20170228.csv')
dat14 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g14_avg1m_20121001_20160714.csv')
dat13 = pd.read_csv('/mnt/data/Checking_GOES/Combined/combined_g13_avg1m_20130607_20170228.csv')


In [ ]:
dat18 = dat18[(dat18['xrsb_flag'] == 0) ]
dat17 = dat17[(dat17['xrsb_flag'] == 0) ]
dat16 = dat16[(dat16['xrsb_flag'] == 0) ]
dat15 = dat15[(dat15['xrsb_flag'] == 0)]
dat14 = dat14[(dat14['xrsb_flag'] == 0)]
dat13 = dat13[(dat13['xrsb_flag'] == 0)]

In [ ]:
dat18 = dat18[(dat18['xrsb_flag'] == 0) & (dat18['xrsa_flag'] == 0)]
dat17 = dat17[(dat17['xrsb_flag'] == 0) & (dat17['xrsa_flag'] == 0)]
dat16 = dat16[(dat16['xrsb_flag'] == 0) & (dat16['xrsa_flag'] == 0)]
dat15 = dat15[(dat15['xrsb_flag'] == 0) & (dat15['xrsa_flag'] == 0)]
dat14 = dat14[(dat14['xrsb_flag'] == 0) & (dat14['xrsa_flag'] == 0)]
dat13 = dat13[(dat13['xrsb_flag'] == 0) & (dat13['xrsa_flag'] == 0)]



In [ ]:
print(dat18.head())

In [ ]:
import pandas as pd

# First, combine all dataframes into one, add a column identifying the instrument for clarity
dat18['gnum'] = 18
dat17['gnum'] = 17
dat16['gnum'] = 16
dat15['gnum'] = 15
dat14['gnum'] = 14
dat13['gnum'] = 13

all_data = pd.concat([dat18, dat17, dat16, dat15, dat14, dat13], axis=0, ignore_index=True)

# Convert 'time' to a datetime type (if not already)
all_data['time'] = pd.to_datetime(all_data['time'])

# For each overlapping time, average flux values
# We'll average 'xrsa_flux' and 'xrsb_flux'
grouped = all_data.groupby('time').agg(
    xrsa_flux_mean=('xrsa_flux', 'mean'),
    xrsb_flux_mean=('xrsb_flux', 'mean'),
    num_instruments=('gnum', 'nunique')
).reset_index()

# Preview the merged data
print(grouped.head(10))
print(f"\nTotal unique timestamps: {len(grouped)}")
print(f"Number of timestamps with more than one instrument: {(grouped['num_instruments'] > 1).sum()}")

# If you want to save or use this merged DataFrame:
# grouped.to_csv('/mnt/data/Checking_GOES/Combined/merged_goes_avg.csv', index=False)
# Start Generation Here


In [ ]:
import os

# Path to the SXR split folders
base_dir = "/mnt/data/NO-OVERLAP/SXR"
split_folders = ['test', 'val', 'train']

# Get all timestamp strings from the grouped dataframe, formatted as they would appear in npy filenames
all_times = grouped['time'].dt.strftime('%Y-%m-%dT%H:%M:%S').tolist()

# Build a set of all .npy files in the split folders (using the specified structure)
existing_times = set()
for split in split_folders:
    folder = os.path.join(base_dir, split)
    if os.path.exists(folder):
        for fname in os.listdir(folder):
            if fname.endswith('.npy'):
                time_str = fname[:-4]  # remove '.npy'
                # The full path would be like: /mnt/data/NO-OVERLAP/SXR/val/2023-03-31T07:43:00.npy
                # We only need the timestamp part (filename minus extension)
                existing_times.add(time_str)

# Find which times from grouped are NOT present in any split folder
missing_times = [t for t in all_times if t not in existing_times]

print(f"Number of grouped times: {len(all_times)}")
print(f"Number of unique SXR files found in all splits: {len(existing_times)}")
print(f"Number missing: {len(missing_times)}")
print("\nFirst 10 missing times:")
print(missing_times[:10])

# Optionally, get full DataFrame of missing rows
missing_rows_df = grouped[grouped['time'].dt.strftime('%Y-%m-%dT%H:%M:%S').isin(missing_times)]

# Count how many grouped times actually have a matching .npy file in any split folder (i.e., are present in existing_times)
matching_times = [t for t in all_times if t in existing_times]
print(f"Number of grouped times with a matching .npy file: {len(matching_times)}")


In [ ]:
import numpy as np

# Directory to save [SXR_A, SXR_B] arrays, as numpy files, keyed by timestamp
ab_target_dir = "/mnt/data/PAPER_SXR_B_CLEAN"
os.makedirs(ab_target_dir, exist_ok=True)

ab_copied_count = 0
ab_not_found = []

# Ensure 'grouped' DataFrame is indexed by 'time' for quicker access if not already
# and that times are formatted consistently
grouped_times_formatted = grouped['time'].dt.strftime('%Y-%m-%dT%H:%M:%S')

for i, row in grouped[grouped_times_formatted.isin(matching_times)].iterrows():
    tstr = row['time'].strftime('%Y-%m-%dT%H:%M:%S')
    try:
        if ab_target_dir == "/mnt/data/PAPER_SXR_B_CLEAN":
            sxr_b = row['xrsb_flux_mean']
            arr = np.array([sxr_b])
        else:
            sxr_a = row['xrsa_flux_mean']
            sxr_b = row['xrsb_flux_mean']
            arr = np.array([sxr_a, sxr_b])
        out_path = os.path.join(ab_target_dir, f"{tstr}.npy")
        np.save(out_path, arr)
        ab_copied_count += 1
    except Exception as e:
        ab_not_found.append((tstr, str(e)))

print(f"Saved {ab_copied_count} [SXR_A, SXR_B] .npy files to {ab_target_dir}.")
if ab_not_found:
    print(f"{len(ab_not_found)} time(s) could not be processed (first 10): {ab_not_found[:10]}")



In [ ]:
import glob
import pandas as pd

# Get all .npy files in the ab_target_dir
csv_target_dir = "/mnt/data/PAPER_SXR_A_B_CLEAN"
all_npy_files = glob.glob(os.path.join(csv_target_dir, "*.npy"))

# Extract timestamps from filenames (remove path and .npy extension)
timestamps = [os.path.splitext(os.path.basename(f))[0] for f in all_npy_files]

# Save to CSV as a single column
df_timestamps = pd.DataFrame({'timestamp': timestamps})
df_timestamps.to_csv(os.path.join(csv_target_dir, 'timestamps.csv'), index=False)

print(f"Saved {len(timestamps)} timestamps to {os.path.join(csv_target_dir, 'timestamps.csv')}.")


In [ ]:
import os
import shutil
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed

# Define source and destination directories
aia_base_dir = "/mnt/data/NO-OVERLAP/AIA"
aia_subfolders = ["test", "train", "val"]
aia_copy_dest = "/mnt/data/PAPER_SXR_A_B_AIA/"

# Make sure destination directory exists
os.makedirs(aia_copy_dest, exist_ok=True)

# Load the timestamps we want, which we just saved as timestamps.csv
timestamps_csv_path = os.path.join("/mnt/data/PAPER_SXR_A_B_CLEAN", "timestamps.csv")
df_timestamps = pd.read_csv(timestamps_csv_path)
timestamps_set = set(df_timestamps['timestamp'].astype(str).tolist())

def find_and_copy(ts):
    """Try to find the .npy for this timestamp in the aia_subfolders, and copy if found."""
    for sub in aia_subfolders:
        src_file = os.path.join(aia_base_dir, sub, f"{ts}.npy")
        if os.path.isfile(src_file):
            dest_file = os.path.join(aia_copy_dest, f"{ts}.npy")
            try:
                shutil.copy2(src_file, dest_file)
                return (ts, True, None)  # copied
            except Exception as e:
                return (ts, False, f"Copy error: {e}")
    return (ts, False, "Not found")

results = []
max_workers = min(16, os.cpu_count() or 8)

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    # Start all tasks (returns futures)
    futures = {executor.submit(find_and_copy, ts): ts for ts in timestamps_set}

    for f in tqdm(as_completed(futures), total=len(futures), desc="Copying AIA files for timestamps"):
        results.append(f.result())

copied_count = sum(1 for r in results if r[1])
missing_files = [r[0] for r in results if not r[1] and r[2] == "Not found"]
other_errors = [(r[0], r[2]) for r in results if not r[1] and r[2] != "Not found"]

print(f"Copied {copied_count} AIA .npy files to {aia_copy_dest}.")
if missing_files:
    print(f"Could not find files for {len(missing_files)} timestamps (first 10): {missing_files[:10]}")
if other_errors:
    print(f"There were {len(other_errors)} copy errors (first 5): {other_errors[:5]}")
